# pipeline

> The headless decomposition pipeline: load a transcription run manifest, then per source
> per pipeline-segment run VAD + forced alignment, build one aligned segment per VAD chunk
> (times offset to source coordinates), commit the spine to the graph, and verify — with
> HITL approval seams between alignment, commit, and the next source.

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
import json
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue, JobStatus

from cjm_transcript_decomp_core.models import (
    DecompConfig, DecompSegment, DecompDocument, DecompManifest,
    FAWord, VADChunk, new_run_id,
)
from cjm_transcript_decomp_core.alignment import (
    map_fa_words_to_text, assign_words_to_chunks, build_segments_from_alignment,
    tier1_alignment_checks,
)
from cjm_transcript_decomp_core.graph import (
    build_graph_payload, commit_graph, verify_document,
)

logger = logging.getLogger(__name__)

In [ ]:
#| export
def field_of(
    result: Any,          # Capability result — dict over the proxy wire, object in-process
    key: str,             # Field name to read
    default: Any = None,  # Fallback when absent
) -> Any:  # The field value or the default
    """Read a field from a dict-or-object capability result.

    Re-implemented here rather than shared (Nth ecosystem copy — pass-2 evidence
    E5: results need a typed wire layer, not per-core tolerance helpers).
    """
    if isinstance(result, dict):
        return result.get(key, default)
    return getattr(result, key, default)

In [ ]:
#| export
async def submit_and_wait(
    queue: JobQueue,   # Started job queue
    instance_id: str,  # Capability instance to invoke
    *,
    timeout: Optional[float] = None,  # Seconds to wait; None = no limit
    **kwargs,          # Forwarded to the capability's execute()
) -> Any:  # The completed job's result payload
    """Submit one capability job, wait for it, and return its result (raise on failure)."""
    job_id = await queue.submit(instance_id, **kwargs)
    job = await queue.wait_for_job(job_id, timeout=timeout)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{instance_id} job {job_id} {job.status}: {job.error}")
    return job.result

In [ ]:
#| export
def load_source_manifest(
    path: str,  # Path to a transcription-core run manifest JSON
) -> Dict[str, Any]:  # Parsed manifest dict
    """Load + lightly validate a transcription-core run manifest.

    Consumed as untyped JSON — the format/version tags are the only interchange
    contract (no shared schema type across cores; manifest-as-interchange, CR-20).
    """
    data = json.loads(Path(path).read_text())
    fmt = data.get("format", "")
    if "transcription-core" not in fmt:
        logger.warning(f"unexpected source manifest format: {fmt!r} (continuing)")
    return data

In [ ]:
#| export
async def analyze_segment_vad(
    queue: JobQueue,
    vad_id: str,          # VAD capability instance id
    audio_path: str,      # Pipeline-segment audio file (model-input WAV)
    force: bool = False,  # Bypass the VAD cache
) -> List[VADChunk]:  # Segment-local VAD chunks, sorted, re-indexed
    """Run VAD on one pipeline-segment audio file -> segment-local VAD chunks.

    Per-segment VAD (not whole-source) is the validated decomp path; it was
    originally chosen to avoid browser Web-Audio limits on hours-long source
    audio in the GUI (a presentation constraint), and holds on the merits here.
    """
    result = await submit_and_wait(queue, vad_id, media_path=audio_path, force=force)
    ranges = field_of(result, "ranges", []) or []
    chunks = []
    for r in ranges:
        chunks.append(VADChunk(
            index=0,
            start_time=float(field_of(r, "start", field_of(r, "start_time", 0.0))),
            end_time=float(field_of(r, "end", field_of(r, "end_time", 0.0))),
        ))
    chunks.sort(key=lambda c: c.start_time)
    for i, c in enumerate(chunks):
        c.index = i
    return chunks

In [ ]:
#| export
async def align_segment(
    queue: JobQueue,
    fa_id: str,           # Forced-alignment capability instance id
    audio_path: str,      # Pipeline-segment audio file (model-input WAV)
    text: str,            # Transcript text for this pipeline segment
    force: bool = False,  # Bypass the FA cache
) -> List[FAWord]:  # Word-level alignment results
    """Run forced alignment on one (segment audio, text) pair -> FA words."""
    result = await submit_and_wait(queue, fa_id, audio=audio_path, text=text, force=force)
    items = field_of(result, "items", []) or []
    words = []
    for it in items:
        if isinstance(it, dict):
            words.append(FAWord.from_wire(it))
        else:
            words.append(FAWord(
                text=str(field_of(it, "text", "")),
                start_time=float(field_of(it, "start_time", 0.0)),
                end_time=float(field_of(it, "end_time", 0.0)),
            ))
    return words

In [ ]:
#| export
async def decompose_source(
    queue: JobQueue,
    cfg: DecompConfig,         # Run configuration
    source: Dict[str, Any],    # One source entry from the transcription manifest
    manifest_path: str,        # Consumed manifest path (for provenance)
    source_index: int,         # Position of this source within the run
    transcriber_name: str,     # Upstream transcriber capability name
) -> Tuple[str, List[DecompSegment]]:  # (source_path, ordered aligned segments)
    """Decompose one source's transcription segments into aligned graph segments.

    Per pipeline segment: re-run VAD on the segment audio (segment-local chunks),
    force-align the segment text, then build one aligned segment per VAD chunk.
    Times are offset to source coordinates; indices accumulate across the source's
    pipeline segments.
    """
    source_path = str(field_of(source, "source_path", ""))
    seg_list = list(field_of(source, "segments", []) or [])
    aligned: List[DecompSegment] = []
    global_index = 0

    for pseg in seg_list:
        model_input = str(field_of(pseg, "model_input_path", ""))
        text = str(field_of(pseg, "text", "") or "")
        seg_start = float(field_of(pseg, "start", 0.0))
        job_id = field_of(pseg, "job_id")
        if not text.strip():
            logger.warning(f"[src {source_index}] empty text at {seg_start:.1f}s; skipping pipeline segment")
            continue

        vad_chunks = await analyze_segment_vad(queue, cfg.vad_plugin, model_input, force=cfg.force)
        fa_words = await align_segment(queue, cfg.fa_plugin, model_input, text, force=cfg.force)
        spans = map_fa_words_to_text(text, fa_words)
        assignments = assign_words_to_chunks(fa_words, vad_chunks)
        text_segments = build_segments_from_alignment(
            text=text, spans=spans, assignments=assignments,
            num_chunks=len(vad_chunks), source_id=job_id, source_provider_id=transcriber_name,
        )
        for w in tier1_alignment_checks(text_segments, vad_chunks):
            logger.warning(f"[src {source_index}] pseg @ {seg_start:.1f}s: {w}")

        for ts, vc in zip(text_segments, vad_chunks):
            aligned.append(DecompSegment(
                index=global_index, text=ts.text,
                start_time=seg_start + vc.start_time, end_time=seg_start + vc.end_time,
                start_char=ts.start_char, end_char=ts.end_char,
                source_job_id=job_id, source_provider_id=transcriber_name,
                vad_chunk_index=vc.index,
            ))
            global_index += 1
        logger.info(f"[src {source_index}] pseg @ {seg_start:.1f}s -> "
                    f"{len(vad_chunks)} chunks, {len(fa_words)} words")
    return source_path, aligned

In [ ]:
#| export
def confirm_seam(
    seam: str,                 # Seam label, e.g. "alignment-review"
    summary_lines: List[str],  # What the operator is being asked to accept
    warnings: List[str],       # Tier-1 warnings (logged prominently)
    assume_yes: bool = False,  # Headless mode: accept without prompting
) -> bool:  # True = proceed, False = operator aborted
    """HITL approval seam in its cheapest viable form (log + optional CLI prompt).

    Per-seam HITL-assist annotation (5 fields):
      1. signal: per-document summary + Tier-1 warnings
      2. deterministic pre-filter: tier1_alignment_checks (no AI)
      3. modality-bridge candidate: spectrogram / word-overlay review (future Tier 2)
      4. authoritative verifier: re-align or re-transcribe-and-compare (future Tier 3)
      5. flywheel capture: accept/abort decisions logged; durable capture is a
         pass-2 seam-contract concern, not solved here

    NOTE: input() blocks the event loop — acceptable because seams sit between
    stages with no jobs in flight; the pass-2 seam contract needs an async shape.
    """
    for line in summary_lines:
        logger.info(f"[{seam}] {line}")
    for w in warnings:
        logger.warning(f"[{seam}] {w}")
    if assume_yes:
        logger.info(f"[{seam}] auto-accepted (assume_yes)")
        return True
    reply = input(f"[{seam}] proceed? [Y/n] ").strip().lower()
    accepted = reply in ("", "y", "yes")
    logger.info(f"[{seam}] {'accepted' if accepted else 'ABORTED'} by operator")
    return accepted

In [ ]:
#| export
def collect_plugin_info(
    manager: PluginManager,   # Manager holding the loaded capabilities
    instance_ids: List[str],  # Instance ids to record
) -> Dict[str, Dict[str, Any]]:  # instance_id -> {name, version, db_path}
    """Record capability identity + data-DB pointers for the run manifest (provenance)."""
    info: Dict[str, Dict[str, Any]] = {}
    for iid in instance_ids:
        meta = (getattr(manager, "plugins", {}) or {}).get(iid)
        if meta is None:
            continue
        manifest = getattr(meta, "manifest", {}) or {}
        info[iid] = {"name": meta.name, "version": getattr(meta, "version", None),
                     "db_path": manifest.get("db_path")}
    return info

In [ ]:
#| export
async def run_decomp(
    manager: PluginManager,        # Manager with VAD + FA + graph capabilities loaded
    queue: JobQueue,               # Started job queue
    cfg: DecompConfig,             # Run configuration
    source_manifest_path: str,     # Transcription run manifest to decompose
    run_id: Optional[str] = None,  # Override run id (default: generated)
) -> DecompManifest:  # Manifest of the documents produced
    """Decompose every source in a transcription run manifest into graph documents.

    Per source: align -> [alignment-review seam] -> build payload ->
    [commit-review seam] -> commit -> verify. An operator abort at any seam stops
    the run; the manifest holds the documents committed so far.
    """
    run_id = run_id or new_run_id()
    src = load_source_manifest(source_manifest_path)
    transcriber_name = (src.get("config", {}) or {}).get("transcriber_plugin", "unknown")

    manifest = DecompManifest(
        run_id=run_id, created_at=time.time(), config=cfg.to_dict(),
        source_manifest=str(source_manifest_path),
        source_format=src.get("format", ""), source_version=src.get("version", ""),
        plugins=collect_plugin_info(manager, [cfg.vad_plugin, cfg.fa_plugin, cfg.graph_plugin]),
    )

    sources = src.get("sources", []) or []
    for i, source in enumerate(sources):
        source_path, aligned = await decompose_source(
            queue, cfg, source, str(source_manifest_path), i, transcriber_name)
        title = Path(source_path).stem or f"document-{i}"

        empty = sum(1 for a in aligned if not a.text.strip())
        warns = [f"{empty}/{len(aligned)} aligned segment(s) have empty text"] if empty else []
        if not confirm_seam("alignment-review",
                            [f"{title}: {len(aligned)} aligned segment(s)"],
                            warns, assume_yes=cfg.assume_yes):
            logger.warning(f"run {run_id}: aborted at source {i} ({source_path})")
            break

        nodes, edges, doc_id, seg_ids = build_graph_payload(
            title, aligned, str(source_manifest_path), media_type=cfg.media_type)

        if not confirm_seam("commit-review",
                            [f"{title}: committing {len(seg_ids)} segment node(s) + "
                             f"{len(edges)} edge(s) to {cfg.graph_plugin}"],
                            [], assume_yes=cfg.assume_yes):
            logger.warning(f"run {run_id}: commit declined at source {i} ({source_path})")
            break

        counts = await commit_graph(queue, cfg.graph_plugin, nodes, edges)
        logger.info(f"[src {i}] committed {counts['nodes']} node(s), {counts['edges']} edge(s)")

        vr = await verify_document(queue, cfg.graph_plugin, doc_id)
        if vr is None:
            logger.error(f"[src {i}] verify: document {doc_id} NOT FOUND in graph")
        else:
            logger.info(f"[src {i}] verify: {'OK' if vr.ok else 'FAILED'} "
                        f"(segments={vr.segment_count}, starts_with={vr.has_starts_with}, "
                        f"next_chain={vr.next_chain_complete}, part_of={vr.part_of_complete}, "
                        f"timing={vr.all_have_timing}, sources={vr.all_have_sources})")

        manifest.documents.append(DecompDocument(
            document_id=doc_id, source_path=source_path, title=title,
            segment_count=len(seg_ids), segment_ids=seg_ids))
    return manifest

In [ ]:
# pipeline import smoke check (no capabilities involved)
assert callable(run_decomp)
assert callable(decompose_source)
_m = load_source_manifest  # symbol present
print("pipeline import checks OK")

pipeline import checks OK
